In [5]:
from pathlib import Path

import pandas as pd

interim = Path("datos/interim")
if not interim.is_dir():
    interim = Path("../datos/interim")

archivos = sorted(interim.glob("carrera_*_interim.csv"))
assert len(archivos) == 6, f"Esperaba 6 CSV y encontré {len(archivos)}"

resultados = pd.concat(
    [pd.read_csv(archivo, dtype=str, keep_default_na=False) for archivo in archivos],
    ignore_index=True,
)

filas_por_carrera = resultados.groupby("id_carrera").size()
display(filas_por_carrera.to_frame("filas"))
print("Total de filas:", len(resultados))

,filas
id_carrera,
carrera_01,1489
carrera_02,929
carrera_03,795
carrera_04,875
carrera_05,659
carrera_06,643


Total de filas: 5390


In [6]:
# Unificamos espacios y mayúsculas para poder comparar los formatos.
puesto = resultados["puesto"].str.strip()
sexo = resultados["puesto_sexo"].str.strip().str.upper()
categoria = (
    resultados["puesto_categoria"]
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)
tiempo = resultados["tiempo_oficial"].str.strip()
dorsal = resultados["dorsal"].str.strip()

categorias_especiales = ["ESPECIAL", "SEN CATEGORÍA", "SEN CATEGORIA"]
formato_categoria = categoria.str.fullmatch(r"[A-Z0-9]+[MF]\s*-\s*[1-9]\d*")
formato_tiempo = tiempo.str.fullmatch(r"(?:\d+:[0-5]\d|\d+):[0-5]\d")

incidencias = pd.DataFrame({
    "nombre_vacio": resultados["nombre_completo"].str.strip().eq(""),
    "dorsal_vacio": dorsal.eq(""),
    "dorsal_repetido": dorsal.ne("") & resultados.assign(
        dorsal=dorsal
    ).duplicated(["id_carrera", "dorsal"], keep=False),
    "fila_duplicada": resultados.drop(columns="pagina_origen").duplicated(keep=False),
    "puesto_invalido": ~puesto.str.fullmatch(r"[1-9]\d*"),
    "puesto_sexo_vacio": sexo.eq(""),
    "puesto_sexo_formato_inesperado": (
        sexo.ne("") & ~sexo.str.fullmatch(r"[MF]\s*-\s*[1-9]\d*")
    ),
    "categoria_vacia": categoria.eq(""),
    "categoria_especial": categoria.isin(categorias_especiales),
    "categoria_formato_inesperado": (
        categoria.ne("") & ~formato_categoria & ~categoria.isin(categorias_especiales)
    ),
    "categoria_ajustada_en_espacios_o_mayusculas": (
        categoria.ne(resultados["puesto_categoria"])
    ),
    "tiempo_vacio": tiempo.eq(""),
    "tiempo_formato_inesperado": tiempo.ne("") & ~formato_tiempo,
    "tiempo_cero": tiempo.str.fullmatch(r"0+(?::00){1,2}"),
})

resumen = incidencias.groupby(resultados["id_carrera"]).sum().T
resumen["total"] = resumen.sum(axis=1)
display(resumen)

id_carrera,carrera_01,carrera_02,carrera_03,carrera_04,carrera_05,carrera_06,total
nombre_vacio,0,0,0,0,0,0,0
dorsal_vacio,0,0,0,0,0,0,0
dorsal_repetido,0,0,0,0,0,0,0
fila_duplicada,0,0,0,0,0,0,0
puesto_invalido,0,0,0,0,0,0,0
puesto_sexo_vacio,0,0,0,0,0,0,0
puesto_sexo_formato_inesperado,3,6,5,0,3,0,17
categoria_vacia,0,0,0,0,0,0,0
categoria_especial,0,0,0,0,0,0,0
categoria_formato_inesperado,3,6,5,0,3,0,17


In [7]:
limpias = resultados.copy()

# Puesto general y puesto por sexo.
limpias["puesto_general"] = puesto.where(
    puesto.str.fullmatch(r"[1-9]\d*")
).astype("Int64")

partes_sexo = sexo.str.extract(
    r"^(?P<sexo>[MF])\s*-\s*(?P<puesto_por_sexo>[1-9]\d*)$"
)
limpias["sexo"] = partes_sexo["sexo"].astype("string")
limpias["puesto_por_sexo"] = partes_sexo["puesto_por_sexo"].astype("Int64")

# Ejemplo: MS2M-1 da categoría MS2, sexo M y puesto 1.
partes_categoria = categoria.str.extract(
    r"^(?P<categoria>[A-Z0-9]+)(?P<sexo_categoria>[MF])"
    r"\s*-\s*(?P<puesto_por_categoria>[1-9]\d*)$"
)
limpias["categoria"] = partes_categoria["categoria"].astype("string")
limpias["sexo_categoria"] = partes_categoria["sexo_categoria"].astype("string")
limpias["puesto_por_categoria"] = partes_categoria["puesto_por_categoria"].astype("Int64")

# NO tiene puesto de categoría, pero su significado está pendiente de confirmar.
puesto_no = categoria.str.extract(r"^NO\s*-\s*([1-9]\d*)$", expand=False)
es_no = puesto_no.notna()
limpias.loc[es_no, "categoria"] = "NO"
limpias.loc[es_no, "puesto_por_categoria"] = puesto_no[es_no].astype("Int64")

# Convertimos MM:SS o HH:MM:SS a segundos.
partes_tiempo = tiempo.str.extract(
    r"^(?:(?P<horas>\d+):(?=[0-5]\d:))?"
    r"(?P<minutos>\d+):(?P<segundos>[0-5]\d)$"
).astype("Int64")

segundos = (
    partes_tiempo["horas"].fillna(0) * 3600
    + partes_tiempo["minutos"] * 60
    + partes_tiempo["segundos"]
)
positivo = segundos.gt(0).fillna(False)
limpias["tiempo_segundos"] = segundos.where(positivo)
limpias["estado_tiempo"] = "invalido"
limpias.loc[tiempo.eq(""), "estado_tiempo"] = "ausente"
limpias.loc[positivo, "estado_tiempo"] = "valido"

# Dos horas es un umbral provisional de revisión.
limpias.loc[segundos.ge(7200).fillna(False), "estado_tiempo"] = "revisar"

# Mostramos únicamente recuentos.
assert len(limpias) == len(resultados)
control = limpias.assign(
    categoria_NO=es_no,
    sexo_sin_dato=limpias["sexo"].isna(),
    tiempo_a_revisar=limpias["estado_tiempo"].eq("revisar"),
)
print(control.groupby("id_carrera")[
    ["categoria_NO", "sexo_sin_dato", "tiempo_a_revisar"]
].sum().to_string())
print("Filas conservadas:", len(limpias))

            categoria_NO  sexo_sin_dato  tiempo_a_revisar
id_carrera                                               
carrera_01             3              3                 0
carrera_02             6              6                 0
carrera_03             5              5                 0
carrera_04             0              0                 0
carrera_05             3              3                 3
carrera_06             0              0                 0
Filas conservadas: 5390


In [8]:
salida = interim / "resultados_limpios_local.csv"
limpias.to_csv(salida, index=False)
print(f"Guardadas {len(limpias)} filas en {salida.name}")

Guardadas 5390 filas en resultados_limpios_local.csv


## Calidad y limpieza de los resultados

Se cargaron 5.390 filas de seis carreras. El perfil por carrera no detectó
nombres o dorsales vacíos, filas duplicadas ni dorsales repetidos dentro de
una misma prueba. Se conservaron las 5.390 filas.

| Problema | Dimensión | Carreras | Filas | Detección | Decisión |
|---|---|---|---:|---|---|
| Puesto por sexo ausente y categoría `NO` | Completitud y validez | 01 (3), 02 (6), 03 (5), 05 (3) | 17 | `—` en el puesto por sexo y patrón `NO-n` en el puesto por categoría | Conservar el código `NO` y su puesto de categoría. Dejar vacíos el sexo y su puesto. |
| Tiempos extremos | Validez | 05 | 3 | Tiempo igual o superior a 7.200 segundos | Conservar el valor y marcarlo como `revisar`. Se usó ese umbral para detectar casos que requieren comprobación. |

Se separaron el sexo, la categoría y sus puestos, y se convirtieron los
tiempos a segundos. Para el análisis de rendimiento se pueden seleccionar
las filas con `estado_tiempo == "valido"`.